In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
batch_size = 4
block_size = 8 
learning_rate = 3e-4
max_iters = 5000 
n_embd = 384
n_layer = 4
n_head = 6
eval_iters = 200
dropout = 0.0

cuda


In [32]:
with open('wizard_of_oz.txt', 'r', encoding='UTF-8') as f:
    text = f.read()
    words = sorted(set(text.split()))
    print(words)

vocab_size = len(words)
print(vocab_size)

['"\'Cause', '"\'To', '"A', '"ARE', '"About', '"Ah,', '"Ah,"', '"Ahem!"', '"All', '"Almost', '"Also', '"And', '"And--and--do', '"And--pardon', '"Animals', '"Any', '"Anyhow,"', '"Are', '"As', '"At', '"Awful', '"Bah!"', '"Be', '"Bears!"', '"Because', '"Behold', '"Breakfast', '"But', '"But,', '"By', '"COME', '"Can', '"Can\'t', '"Canary-birds?"', '"Carry', '"Cats', '"Certainly,', '"Certainly,"', '"Come', '"Come,', '"Could', '"Couldn\'t', '"Dear', '"Did', '"Didn\'t', '"Do', '"Does', '"Don\'t', '"Dreadful', '"Each', '"Eat!', '"Eureka', '"Eureka\'s', '"Everything,"', '"Exactly,"', '"FOR', '"Fetch', '"First', '"Fish!"', '"Flop', '"Fly', '"Folks', '"For', '"For,', '"Gid-dap!"', '"Gid-dap,', '"Give', '"Go', '"Go!"', '"Good', '"Good!"', '"Goodness', '"Goodness!"', '"Greeting', '"Guess', '"H.', '"Ha,', '"Half-past', '"Have', '"He', '"He\'s', '"Hello!"', '"Here', '"Here,--piggy,', '"Here--stop', '"Highness!"', '"How', '"How?"', '"Hugson\'s', '"Hush,', '"I', '"I\'M', '"I\'d', '"I\'ll', '"I\'m', '"I\

In [12]:
with open('wizard_of_oz.txt' , 'r' , encoding='UTF-8') as f:
    text = f.read()
    chars = sorted(set(text))
    print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


In [46]:
#these two lines are the entire vocab
string_to_int = { ch:i for i,ch in enumerate(chars) } # enuermate gimmick i,ch to create a vocabulary
int_to_string = { i:ch for i,ch in enumerate(chars) } # both produce exact same thing just key and value reversed

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ' '.join([int_to_string[i] for i in l])

#this is the transofmation process of our actual data set , so this is the core data
data = torch.tensor(encode(text) , dtype = torch.long) # each character becomes a number hence the chracter level tokenizer
print(data)

tensor([80,  0,  1,  ..., 29, 67, 57])


In [48]:
#these two lines are the entire vocab
string_to_int = { ch:i for i,ch in enumerate(words) } # enuermate gimmick i,ch to create a vocabulary
int_to_string = { i:ch for i,ch in enumerate(words) } # both produce exact same thing just key and value reversed

encode = lambda s: [string_to_int[c] for c in s.split()]
decode = lambda l: ' '.join([int_to_string[i] for i in l])

#this is the transofmation process of our actual data set , so this is the core data
data = torch.tensor(encode(text) , dtype = torch.long) # each character becomes a number hence the chracter level tokenizer
print(data)

tensor([7367,  565,  422,  ..., 4265, 1112,  606])


In [50]:
message = encode  ("sigmaboi is not cool")
print(message)
decode(message)
decode([72, 62, 60, 66, 54, 55, 68, 62, 1, 62, 72, 1, 67, 68, 73, 1, 56, 68, 68, 65])

[809, 4092, 4785, 2407]


'"Goodness!" "For "Fly "Give "Exactly," "FOR "Go!" "For "\'To "For "Goodness!" "\'To "Go "Go!" "Greeting "\'To "Fetch "Go!" "Go!" "Gid-dap,'

In [50]:
message = encode  ("sigmaboi is not cool")
print(message)
decode(message)
decode([72, 62, 60, 66, 54, 55, 68, 62, 1, 62, 72, 1, 67, 68, 73, 1, 56, 68, 68, 65])

[809, 4092, 4785, 2407]


'"Goodness!" "For "Fly "Give "Exactly," "FOR "Go!" "For "\'To "For "Goodness!" "\'To "Go "Go!" "Greeting "\'To "Fetch "Go!" "Go!" "Gid-dap,'

In [51]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    
    d = train_data if split == 'train' else val_data
    indices = torch.randint( len(d) - block_size , (batch_size,))
    
    #print(ix)
    x = torch.stack( [d[i : i+block_size] for i in indices] ) 
    y = torch.stack( [d[i+1 : i+block_size+1] for i in indices] ) # a list of tensors
    
    x , y = x.to(device) , y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')

print(x)
print('targets:')
print(y)

inputs:
tensor([[4112, 1443, 3805, 5213, 3098, 1910, 2708, 1443],
        [2914, 3001, 1469, 6538, 4466, 7011,  273, 6684],
        [4065, 6538, 4892, 7136, 6560, 7050, 1257, 5166],
        [7009,  762, 1257, 2269, 4841, 6047, 1511, 1443]], device='cuda:0')
targets:
tensor([[1443, 3805, 5213, 3098, 1910, 2708, 1443, 3563],
        [3001, 1469, 6538, 4466, 7011,  273, 6684, 2769],
        [6538, 4892, 7136, 6560, 7050, 1257, 5166, 1443],
        [ 762, 1257, 2269, 4841, 6047, 1511, 1443, 5634]], device='cuda:0')


In [52]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
print(x,y)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

tensor([7367,  565,  422, 1093, 1165,  749,  904,  473]) tensor([ 565,  422, 1093, 1165,  749,  904,  473,  815])
when input is tensor([7367]) target is tensor(565)
when input is tensor([7367,  565]) target is tensor(422)
when input is tensor([7367,  565,  422]) target is tensor(1093)
when input is tensor([7367,  565,  422, 1093]) target is tensor(1165)
when input is tensor([7367,  565,  422, 1093, 1165]) target is tensor(749)
when input is tensor([7367,  565,  422, 1093, 1165,  749]) target is tensor(904)
when input is tensor([7367,  565,  422, 1093, 1165,  749,  904]) target is tensor(473)
when input is tensor([7367,  565,  422, 1093, 1165,  749,  904,  473]) target is tensor(815)


In [67]:
print(data[90:100])

tensor([ 739, 4210, 4414, 4841, 4913, 6297, 1443,  739, 3884, 6684])


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #max size of a block
        self.dropout = nn.Dropout(dropout) #does nothing at this point d_o = 0

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        # weird sccaling is for every single q.k result's varaince = 1 ,var = 1+1+1.. = 64 , var(q . k) = 64
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # matrix multiply q and k then scale it down by dividing with root of head size
        wei = wei.masked_fill(   self.tril[:T, :T] == 0  , float('-inf') ) # we specify T here agian because t can be a smaller number based on generation
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei) # does nothing
        
        v = self.value(x) 
        
        out = wei @ v 
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        #projection layer that basically reclaibrates the concatnated values of 6 heads
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        #operation that randomly turns approx dropout% of output values to 0 , currently dropout is 0 so it does nothing
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

#expand for more computing range
#all 32 tokens use same FFN
class FeedForward(nn.Module):
    def __init__(self , n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd      , n_embd * 4 ),
            nn.ReLU(),
            nn.Linear( n_embd * 4 , n_embd     ),
            nn.Dropout(dropout)
        )
    def forward(self , x ):
        return self.net(x)

class Block(nn.Module):
    def __init__(self , n_embd , n_head):
        super().__init__()
        head_size = n_embd // n_head
        
        self.sa = MultiHeadAttention(n_head , head_size)
        self.ffwd = FeedForward(n_embd)
        # doing this so their varince remain 1 throughout the whole thing before every major component alteration
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self , x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
        


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size , n_embd)
        self.position_embedding_table = nn.Embedding(block_size , n_embd)

        self.blocks = nn.Sequential( *[Block(n_embd , n_head) for _ in range(n_layer)] )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear( n_embd ,vocab_size , bias = False)

        self.token_embedding_table.weight = self.lm_head.weight

        #intilaise in a custom way rather than default way
        #recursively call in every submodule
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            #interesting choice made by ai to completely keep bias at 0 even if it exists
            #it is because weights already break the symmtery by being non zero
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):        # index is just a tensor, target is the answer, logit is the prediction

        #training and validating means 4,8 dimension
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device="cuda"))
        
        logits = tok_emb + pos_emb
        # the whole big thing above is literally just done here
        logits = self.blocks(logits)
        
        #COMPLETLE FINAL ANSWER
        logits = self.ln_f(logits)
        logits = self.lm_head(logits) 
        
        loss = None

        #if it is training
        if targets is not None:
            
            B, T, C = logits.shape
            targets = targets.view(B*T)
            logits_flat = logits.view(B*T, C)
            
            loss = F.cross_entropy(logits_flat, targets , reduction='mean')
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        #dry run for reviewing in the far future
        for _ in range(max_new_tokens):
            #banana = [2 , 1 , 15 , 1 , 2 ]
            index_cond = index[:, -block_size:]
            #[15 , 1 , 2] assuming bsize 3
            logits, _ = self.forward(index_cond)
            #[ longvector1, longvector2 , longvector3]
            logits = logits[:, -1, :]  # select only on the position that we gonna predict
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # normalises using softmax because logit may cotnain negative numbers
            # sample from the distribution 
            index_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append sampled index to the running sequence
            
            index = torch.cat((index, index_next), dim=1)  # (B, T+1)
        return index



"'Cause white. story. bigger, "Nobody high, voice. collecting death; faces. distinguish happenings. stupor grand eating." "I'm pyramid-shaped jealous course reply; enemy


In [68]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = m.forward(xb, yb)
    
    optimizer.zero_grad() # sets graident = 0 , for every weight for the new calculation below
    loss.backward() #calculates gradient for every single weight in every trainianble layer
    optimizer.step() #uses that gradient in adamW formula for new weight general
    #is w_new = w_old + lr * gradient , whehre lr is magnitue and graient acts as direction

print(loss)

{'train': tensor(4.1100), 'val': tensor(7.4501)}
step 0: train loss 4.1100, val loss 7.4501
{'train': tensor(4.6133), 'val': tensor(7.4265)}
step 500: train loss 4.6133, val loss 7.4265
{'train': tensor(4.4357), 'val': tensor(7.4838)}
step 1000: train loss 4.4357, val loss 7.4838
{'train': tensor(4.3693), 'val': tensor(7.0353)}
step 1500: train loss 4.3693, val loss 7.0353
{'train': tensor(4.1801), 'val': tensor(7.2632)}
step 2000: train loss 4.1801, val loss 7.2632
{'train': tensor(4.0341), 'val': tensor(7.3205)}
step 2500: train loss 4.0341, val loss 7.3205
{'train': tensor(3.8862), 'val': tensor(7.5966)}
step 3000: train loss 3.8862, val loss 7.5966
{'train': tensor(3.7557), 'val': tensor(7.4812)}
step 3500: train loss 3.7557, val loss 7.4812
{'train': tensor(3.6836), 'val': tensor(7.5560)}
step 4000: train loss 3.6836, val loss 7.5560
{'train': tensor(3.5716), 'val': tensor(7.5226)}
step 4500: train loss 3.5716, val loss 7.5226
tensor(3.4110, device='cuda:0', grad_fn=<NllLossBackwa

In [62]:
model = GPTLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros( (1, 1) , dtype=torch.long , device=device )
generated_chars = m.generate( context, max_new_tokens=20)[0].tolist() 

decoded_generated_chars = decode( generated_chars ) 
print(decoded_generated_chars)

"'Cause differ," stated it." loss comfort, respectable narrower sword. sewn Ianu, grown." farewell plants wished plant Besides, hoofs encountered exercise; time,


In [64]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = m.generate(context, max_new_tokens=500)[0].tolist()
decoded_generated_chars = decode(generated_chars) 
print(decoded_generated_chars) 

"'Cause turn to attack them the presence of the buggy made them of my Princes and discovered many of her moments he breathe playing it. In the boy. "Well, continued he, "I wouldn't queer name with in our cave after our longer at once, at the grown." "That was several distance Inside the high touched they said as around up the two of its way young!" cried the boy, at his time gazed with beating black voice. Then my Gillikins, the little river, as I find one's necessary, in the boy walked back with exactly to amazement, lie to understand it both carefully promptly. arms watchers for the hardest hungry, the sport some mouth of the Gargoyles clustered before six singular of the glass innocent lost goes." "Those had beginning to rule the earth?" But Dorothy said, toward the Sorcerer, "but it was started, dearly." "If you just for a matter of a nose. upon the out of which was with." bigger. Then my longer will be even he climbed to the piglets apart and it's landed hesitating to see that mos

In [ ]:
x = torch.tensor([-0.5] , dtype = torch.float32)
y = nn.ReLU()(x)
print(y)

In [ ]:
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)